In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
# Install the CVAT's fork of Datumaro.
# It has an improved and more flexible YOLO Ultralytics exporter.
# It requires Rust!

# To install Rust, either:
# curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh
# or
# sudo apt install rustup

# To install CVAT's Datumaro fork:
# pip install "datumaro[default] @ git+https://github.com/cvat-ai/datumaro"

In [ ]:
# Standard Library imports
from pathlib import Path
import sys

# External imports
import cv2
from ultralytics import YOLO
from datumaro.components.annotation import AnnotationType
from deep_sort_realtime.deepsort_tracker import DeepSort

# Local imports
from convert_utils import (
    DatumaroDatasetBuilder,
    get_frame_chunks_df,
    load_masks,
    load_annotations,
    load_categories,
    load_errors_df,
    extract_error_frames,
)
sys.path.append(
    r"C:\Users\Manuel\Desktop\Documentos\1.PROJECTS\COMPUTER VISION\TRACKING\SORT_TRACKER"
)
# Local imports
from sort import SortTracker, draw_tracking_box

In [ ]:
ROOT_PATH = Path(r"F:\DATASETS\GIL LAB\Example SAM2")


class Config:
    # Minimum bounding box area required for a blob to be considered
    # TODO: Calculate this threshold based on data
    area_threshold = 200  # px

    min_size_threshold = 20

    # Use 4 for focal follow data, 5 for stationary data.
    # TODO: standardize the use of N zeros when extracting data, maybe 8?
    number_of_zeros = 8

    errors_csv_filepath = ROOT_PATH / "SAM2_errors.csv"

    # # Stationary files
    # RIGHT GX050100
    # DATA_PATH = ROOT_PATH / "stationary" / "GX050100"
    # annotations_filepath = (
    #     ROOT_PATH
    #     / "stationary"
    #     / "MLM_051524_site3_east_A_Right_GX050100_annotations.npy"
    # )
    # masks_filepath = (
    #     ROOT_PATH / "stationary" / "MLM_051524_site3_east_A_Right_GX050100_masks.pkl"
    # )

    # LEFT GX056267_frames
    DATA_PATH = ROOT_PATH / "stationary" / "GX056267_frames2"
    annotations_filepath = (
        ROOT_PATH
        / "stationary"
        / "MLM_051524_site3_east_B_Left_GX056267_annotations.npy"
    )
    masks_filepath = (
        ROOT_PATH / "stationary" / "MLM_051524_site3_east_B_Left_GX056267_masks.pkl"
    )


    # Focal follow files
    # DATA_PATH = ROOT_PATH / "focal_follow" / "MH_SR_062523_6_L"
    # annotations_filepath = (ROOT_PATH / "focal_follow" / "MH_SR_062523_6_L_annotations.npy")
    # masks_filepath = ROOT_PATH / "focal_follow" / "MH_SR_062523_6_L_masks.pkl"

config = Config()

### Original frame number:

- **The annotations file:** uses the original (raw) frame number.

### Extracted frame index:
- **The error file:** Uses the extracted frame index (starts at 0).
- **The filenames of the extracted frames:** May start at 0001.jpg, or may start at a random number, due to different extraction methods for different projects. If possible, you should use the index of the extracted frames, not the file name.
- **The masks dict:** Uses the extracted frame index.

In [ ]:
masks = load_masks(config.masks_filepath)
annotations_df = load_annotations(config.annotations_filepath)

In [ ]:
annotations_df

In [ ]:
annotations_df.ObjType.unique()

In [ ]:
label_categories = load_categories(annotations_df)
label_categories[AnnotationType.label]

In [ ]:
video_id = "_".join(config.annotations_filepath.stem.split("_")[:-1])
video_id

In [ ]:
chunked_df = get_frame_chunks_df(annotations_df)
chunked_df

In [ ]:
video_errors_df = load_errors_df(config.errors_csv_filepath, video_id)
video_errors_df.error_type.unique()

In [ ]:
all_error_frames = extract_error_frames(video_errors_df)
all_error_frames

In [ ]:
classifier = YOLO(r"G:\UPWORK\Upwork 2025\Gil Lab\runs\classify\train\weights\best.pt")

In [ ]:
output_filename = "tracker.mp4"
video_writer = cv2.VideoWriter(
    filename=output_filename,
    fourcc=cv2.VideoWriter_fourcc(*"mp4v"),  # *"MPEG", "MJPG", "mp4v", "FMP4"
    fps=10,
    frameSize=(3840, 2160),
    isColor=True,
)

In [ ]:
background_subtractor = cv2.createBackgroundSubtractorMOG2(detectShadows=True)

In [ ]:
# tracker = SortTracker(max_cycles_without_update=3, min_updates=2, verbose=False, kalman_dt=10.0)
# embedder='clip_ViT-B/16', 'torchreid'
# tracker = DeepSort(max_age=50, nn_budget=100, n_init=2, embedder='clip_ViT-B/16', max_iou_distance=1.0, max_cosine_distance=0.1, gating_only_position=True)

builder = DatumaroDatasetBuilder(
    masks=masks,
    all_error_frames=all_error_frames,
    chunked_df=chunked_df,
    label_categories=label_categories,
    classifier=classifier,
    classifier_conf=0.25,
    target_class="correct_fish_mask",
    video_writer=video_writer,
    #tracker=tracker,
    start_frame=1009,
    max_frames=10,
    # background_subtractor=background_subtractor,
    
    images_path=config.DATA_PATH,
    filename_num_zeros=config.number_of_zeros,
    area_threshold=config.area_threshold,
    min_size_threshold=config.min_size_threshold,
    verbose=True,
    debug=True
)
dataset = builder.build()

# TODO create a video based on the built dataset rather than creating it on the fly

In [ ]:
dataset

In [ ]:
unannotated_items = [item for item in dataset if not item.annotations]
print(f"Number of unnanotated frames: {len(unannotated_items)}")

In [ ]:
dataset.export(str(config.DATA_PATH.parent / "dataset_cvat"), format="cvat")

In [ ]:
dataset.export(
    str(config.DATA_PATH.parent / "dataset_yolo"),
    format="yolo_ultralytics_detection",
    add_path_prefix=False,
    save_media=True,
)